# 소재 데이터베이스 실습

**Materials Project · OQMD · ICSD · AFLOW**

계산·실험 소재 데이터를 모아 검색과 내려받기를 제공하는 공개 데이터베이스.

소재 분야에서 이해하기: 후보 조성의 계산 물성을 먼저 조회해 실험 대상을 좁힌다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 공개 데이터베이스의 구조를 흉내내기

API 키가 필요 없도록, 공개 DB와 같은 형태의 작은 표를 만들어 질의 연습을 합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

import pandas as pd

elements = ['Li', 'Na', 'Mg', 'Al', 'Si', 'Ti', 'Fe', 'Co', 'Ni', 'O']
rows = []
for index in range(300):
    picked = rng.choice(elements[:-1], size=rng.integers(1, 4), replace=False)
    formula = ''.join(sorted(picked)) + 'O' + str(int(rng.integers(1, 4)))
    rows.append({
        'material_id': 'syn-%03d' % index,
        'formula': formula,
        'nelements': len(picked) + 1,
        'band_gap_eV': float(np.clip(rng.gamma(2.0, 0.8), 0, 8)),
        'formation_energy_per_atom_eV': float(rng.normal(-1.6, 0.7)),
        'energy_above_hull_eV': float(np.abs(rng.exponential(0.05))),
        'is_experimentally_observed': bool(rng.random() < 0.35),
    })
catalogue = pd.DataFrame(rows)
print(catalogue.head())
print('\n총 %d개 항목' % len(catalogue))

## 2. 스크리닝 질의

"안정하고, 밴드갭이 태양전지에 적합하고, 실험 보고가 있는" 후보를 찾습니다.

In [ ]:
query = catalogue[(catalogue.energy_above_hull_eV < 0.03)
                  & catalogue.band_gap_eV.between(1.1, 1.8)
                  & catalogue.is_experimentally_observed]
print('조건을 만족하는 후보 %d개' % len(query))
print(query[['material_id', 'formula', 'band_gap_eV', 'energy_above_hull_eV']].head(10))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(catalogue.band_gap_eV, bins=40); axes[0].set_xlabel('band gap (eV)')
axes[0].axvspan(1.1, 1.8, color='green', alpha=0.2)
axes[1].scatter(catalogue.energy_above_hull_eV, catalogue.band_gap_eV, s=8)
axes[1].set_xlabel('energy above hull (eV/atom)'); axes[1].set_ylabel('band gap (eV)')
plt.tight_layout(); plt.show()

## 3. 실제 데이터베이스에 연결할 때

아래는 실행하지 않는 예시입니다. API 키를 발급받아 본인 환경에서 사용하세요.
계산값은 계산 조건(범함수·수렴 조건)에 따라 달라지므로 출처와 조건을 함께 기록해야 합니다.

In [ ]:
snippet = """
# pip install mp-api
from mp_api.client import MPRester

with MPRester("YOUR_API_KEY") as mpr:
    docs = mpr.materials.summary.search(
        band_gap=(1.1, 1.8),
        energy_above_hull=(0, 0.03),
        fields=["material_id", "formula_pretty", "band_gap", "energy_above_hull"],
    )
print(len(docs))
"""
print(snippet)
print('키를 노트북에 직접 적어 공유하지 마세요. Colab 의 Secrets 기능이나 환경변수를 쓰세요.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#materials-database)을 여세요.